## Mapping the Questions in Eurobarometer Survey:

In [ ]:
# this will become a function:
import pandas as pd

df = pd.read_excel('./eurobarometer_data/eb_105.xlsx', sheet_name='Content', header=4)
df = df.drop(columns={'Question French'})
df = df.drop(df.index[0])
df = df.replace(',', '', regex=True)
# ask how to make lower case in rows just for one column;
df.head()
#df.to_csv('map_questions.csv', index=False)

,Sheet,Question English
1,D70,D70. On the whole are you very satisfied fairl...
2,D70a,D70a. On the whole are you very satisfied fair...
3,D71_1,D71.1. When you get together with friends or r...
4,D71_2,D71.2. When you get together with friends or r...
5,D71_3,D71.3. When you get together with friends or r...


## Mapping the Answers in Eurobarometer Survey

In order to avoid loading to PostgreSQL tables with too long strings for column names (>63 symbols), I came upp with an approach of mapping all the questions / answers and creating a `.csv` file with a 'map' of all the columns.

Each column is (by now) an answer to a particular question in a survey.
So I need to save the information about:
- which question it was (tab name in Excel file) = question id
- what was the exact text of the question (string)
- amount of answers
- exact answers (strings)
- id for the answers (1,2,3..) + question id

Example | In Eurobarometer 105 we have:
|Tab/Question Id|Question|Number of answers|Exact Answers|Answer Id|
|---|---|---|---|---|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Very Satisfied (1)|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Failty Satisfied (2)
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Not Very Satisfied (3)|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Not at all Satisfied (4)|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Don't Know (5)|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Total Satisfied (6)|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Total Not Satisfied (7)|

In [1]:
import pandas as pd
from retrieve_eurobarometer import read_eurobarometer

In [5]:
file_path_eb = './eurobarometer_data/eb_105.xlsx'
dict_eb = read_eurobarometer(file_path_eb)
df = dict_eb['D70']
df.head()

,<<Back to content,Unnamed: 1,UE27\nEU27,BE,BG,CZ,DK,DEW,DE,DEE,...,MK,ME,RS,AL,MD,UK,BA,XK,CY_TCC,GE
0,NaN,Total,26415,1014,1015,1036,1001,1220,1515,295,...,1017,545,1025,1005,1014,1037,1013,1015,516,1002.00
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Très satisfait(e),5953,245,97,186,670,310,366,56,...,127,22,134,161,137,355,210,282,141,192.00
3,NaN,Very satisfied,0.23,0.24,0.1,0.18,0.67,0.26,0.24,0.19,...,0.13,0.04,0.13,0.16,0.14,0.34,0.21,0.28,0.27,0.19
4,NaN,Plutôt satisfait(e),16883,652,574,733,305,797,984,187,...,641,412,471,574,581,580,583,565,313,465.00


In [20]:
df_test = df.copy()
df_test = df_test.drop(columns={'<<Back to content','UE27\nEU27', 'UE27\\nEU27'},errors='ignore').copy()
# drop first two rows:
df_test = df_test.drop(df_test.index[:2])
df_test.reset_index()
# drop all rows with French / absolute numbers
df_test = df_test.iloc[1::2]
# turn all columns to numeric
num_cols = df_test.columns.drop('Unnamed: 1')
df_test[num_cols] = df_test[num_cols].apply(pd.to_numeric, errors='coerce')
# ret index for the answers:
df_test.set_index('Unnamed: 1',inplace=True)
# flip the table:
df_test = df_test.T
# name the new index column:
df_test.index.name = 'country'
# drop all rows which are not in the list of the EU countries
# df_test = df_test[df_test.index.isin(eu_countries)]
# reset index:
df_test.reset_index(inplace=True)
df_test.columns.name = None

KeyError: "['Unnamed: 1'] not found in axis"

In [11]:
mapped_columns = []
for col in df_test.columns:
    if col == 'country':
        mapped_columns.append(col)
    else:
        col_str = str(col).lower().strip()
        col_str = col_str.replace("'", "").replace(",", "")
        col_str = col_str.replace(":", "").replace(" ", "_")
        col_str = col_str.replace("(", "").replace(")", "")
        #col_str = re.sub(r'_+', '_', col_str)  # collapses ___ down to _
        mapped_columns.append(col_str)
mapped_columns

['country',
 'very_satisfied',
 'fairly_satisfied',
 'not_very_satisfied',
 'not_at_all_satisfied',
 'dont_know',
 'total_satisfied',
 'total_not_satisfied']

In [ ]:
# 2. Define your question details
q_id = "Q1"
q_text = "How satisfied are you with our service?"

# 3. Create the dictionary of lists
# We use list comprehensions to repeat the question info and generate answer IDs
data_dict = {
    "question_id": [q_id] * len(options),
    "question_text": [q_text] * len(options),
    # Generates: Q1_1, Q1_2, Q1_3, etc.
    "answer_id": [f"{q_id}_{i+1}" for i in range(len(options))],
    "answer_text": options
}

# 4. Convert the dictionary into a DataFrame
df = pd.DataFrame(data_dict)

# View the result
print(df)

In [ ]:
# now i need to build a function from it:
# it will need a sheet name


